In [2]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

# CURVAS

data is corrupted. Nothing i can do here.

In [ ]:
DATASET_NAME = "CURVAS"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



15:08:35 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:82


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver'
]

In [4]:
description = """
This repository houses the publicly available training dataset for the Head and Neck Tumor Segmentation for MR-Guided Applications (HNTSMRG) 2024 Challenge.

Patient cohorts correspond to patients with histologically proven head and neck cancer who underwent radiotherapy (RT) at The University of Texas MD Anderson Cancer Center. The cancer types are predominately oropharyngeal cancer or cancer of unknown primary. Images include a pre-RT T2w MRI scan (1-3 weeks before start of RT) and a mid-RT T2w MRI scan (2-4 weeks intra-RT) for each patient. Segmentation masks of primary gross tumor volumes (abbreviated GTVp) and involved metastatic lymph nodes (abbreviated GTVn) are provided for each image (derived from multi-observer STAPLE consensus).
"""

In [ ]:
source = ZenodoSource(
    file_type="nifti",
    record_id="13767408",
    description=description,
    post_download=["unzip"]
)

In [6]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
print(dataset_index_path)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

indexed_datasets/HNTSMRG 2024

In [7]:

print(indexed_datasets.source_config(DATASET_NAME))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    post_download=['unzip'],
    source='zenodo',
    record_id='11199559',
    filenames=None,
    description='\nThis repository houses the publicly available training dataset for the Head and Neck Tumor 
Segmentation for MR-Guided Applications (HNTSMRG) 2024 Challenge.\n\nPatient cohorts correspond to patients with 
histologically proven head and neck cancer who underwent radiotherapy (RT) at The University of Texas MD Anderson 
Cancer Center. The cancer types are predominately oropharyngeal cancer or cancer of unknown primary. Images include
a pre-RT T2w MRI scan (1-3 weeks before start of RT) and a mid-RT T2w MRI scan (2-4 weeks intra-RT) for each 
patient. Segmentation masks of primary gross tumor volumes (abbreviated GTVp) and involved metastatic lymph nodes 
(abbreviated GTVn) are provided for each image (derived from multi-observer STAPLE consensus).\n'
)

In [ ]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
print(temp_data_path.absolute())
downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

/home/gpudual/bhklab/declan/med-image_index/temp_data/HNTSMRG 2024

In [ ]:
import zipfile
from glob import glob
import os

def extract_zip(zip_path, extract_to=None):
    """
    Extract a zip file to a specified directory
    
    Args:
        zip_path: Path to the zip file
        extract_to: Directory to extract to (default: same directory as zip)
    """
    if extract_to is None:
        extract_to = os.path.dirname(zip_path)
    
    # Create extraction directory if it doesn't exist
    os.makedirs(extract_to, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"Extracted {zip_path} to {extract_to}")

for zip_file in glob(f"{temp_data_path}/*.zip"):
    print(zip_file)
    # Get the base name without extension to create a folder with the same name
    zip_name = os.path.splitext(os.path.basename(zip_file))[0]
    extract_to = os.path.join(f"{temp_data_path}", zip_name)
    extract_zip(zip_file, temp_data_path)

In [ ]:
# We have two different scan and mask patterns that i need to capture. We are going to do 2 crawls and combine the indexes.

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="HNTSMRG24_train/{patient_number}/{scan_time}/{name}_T2.nii.gz",
    mask_name_pattern="HNTSMRG24_train/{patient_number}/{scan_time}/{name}_mask.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=True,
)
crawler.crawl()




Parsing 900 files:  51%|██████████████████████████████████████████████████████████████████████████████████████████▍                                                                                       | 457/900 [00:30<00:34, 12.93it/s]11:46:06 [warning  ] Mask /home/gpudual/bhklab/declan/med-image_index/temp_data/HNTSMRG 2024/HNTSMRG24_train/21/midRT/21_midRT_mask.nii.gz is empty. [imgtools] call=parse_niftis._introspect:239
11:46:07 [warning  ] Mask /home/gpudual/bhklab/declan/med-image_index/temp_data/HNTSMRG 2024/HNTSMRG24_train/25/midRT/25_midRT_mask.nii.gz is empty. [imgtools] call=parse_niftis._introspect:239
11:46:08 [warning  ] Mask /home/gpudual/bhklab/declan/med-image_index/temp_data/HNTSMRG 2024/HNTSMRG24_train/29/midRT/29_midRT_mask.nii.gz is empty. [imgtools] call=parse_niftis._introspect:239
Parsing 900 files:  65%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍                                         

In [ ]:

index_df = pd.read_csv(indexed_datasets.path / DATASET_NAME / "index.csv")
index_df



,patient_number,scan_time,name,filepath,file_type,reference_id,class,hash,size,ndim,...,mask.feret_diameter,mask.roundness,mask.flatness,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,is_registered
0,10,midRT,10_midRT,HNTSMRG24_train/10/midRT/10_midRT_T2.nii.gz,scan,10_midRT_10_midRT,MedImage,e8e14d254c581cd759fef50b0d54b126c24da149,"(512, 512, 76)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10,midRT,10_midRT,HNTSMRG24_train/10/midRT/10_midRT_mask.nii.gz,mask,10_midRT_10_midRT,Mask,2bdaf929ea9b48c200053952ea4a0f1875146d7e,"(512, 512, 76)",3,...,57.010964,0.836833,1.923633,1.185530,14.438213,2619.610821,"(17.639697402822986, 33.93230573452438, 40.227...",2.0,HNTSMRG24_train/10/midRT/10_midRT_T2.nii.gz,NaN
2,10,preRT,10_preRT,HNTSMRG24_train/10/preRT/10_preRT_T2.nii.gz,scan,10_preRT_10_preRT,MedImage,83d8db42388c09b9b41c3f980ca87290a2668778,"(512, 512, 75)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10,preRT,10_preRT,HNTSMRG24_train/10/preRT/10_preRT_mask.nii.gz,mask,10_preRT_10_preRT,Mask,65566f1960f7b4bf71264b3442412063e151f6e8,"(512, 512, 75)",3,...,69.631171,0.659162,2.072388,1.676271,14.746881,2732.815072,"(15.27453786193793, 31.654766112668213, 53.061...",2.0,HNTSMRG24_train/10/preRT/10_preRT_T2.nii.gz,NaN
4,101,midRT,101_midRT,HNTSMRG24_train/101/midRT/101_midRT_T2.nii.gz,scan,101_midRT_101_midRT,MedImage,afcd8a4d1bdc913d3c22fe232c8935762a639b77,"(512, 512, 73)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,95,midRT,95_preRT,HNTSMRG24_train/95/midRT/95_preRT_mask_registe...,mask,95_midRT_95_preRT_registered,Mask,ad44969cff59873b27926bed9c4b114d5b89c538,"(768, 768, 116)",3,...,62.248508,0.584874,1.329426,1.772212,17.839432,3999.188850,"(24.385387992901403, 32.41856660529757, 57.452...",3.0,HNTSMRG24_train/95/midRT/95_preRT_T2_registere...,registered
896,96,midRT,96_preRT,HNTSMRG24_train/96/midRT/96_preRT_T2_registere...,scan,96_midRT_96_preRT_registered,MedImage,4edcae7ef36c696019cddeec0e24bd6e507a8781,"(512, 512, 75)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,registered
897,96,midRT,96_preRT,HNTSMRG24_train/96/midRT/96_preRT_mask_registe...,mask,96_midRT_96_preRT_registered,Mask,3c4c62c1cd41c16956fd2af3aeed79e41aae0e01,"(512, 512, 75)",3,...,68.836763,0.544478,1.775101,3.710939,8.743070,960.589361,"(7.704017074688655, 13.6754109042931, 50.74861...",2.0,HNTSMRG24_train/96/midRT/96_preRT_T2_registere...,registered
898,99,midRT,99_preRT,HNTSMRG24_train/99/midRT/99_preRT_T2_registere...,scan,99_midRT_99_preRT_registered,MedImage,fc7cba8c4b17b047b6d86c5d11b3913f0d119f89,"(512, 512, 81)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,registered


In [37]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "MR"
    else:
        return "SEG"
    
def map_roi(row):
    if row["file_type"] == "scan":
            return ""
    else:
        return "GTVp,GTVn"

def map_registration(row):
     if pd.isna(row["is_registered"]):
          return 0
     else: 
        return 1
    
index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["ROINames"] = index_df.apply(map_roi, axis=1)

index_df["is_registered"] = index_df.apply(map_registration, axis=1)

index_df["BodyPartExamined"] = "OROPHARYNX"




In [38]:
index_df

,patient_number,scan_time,name,filepath,file_type,reference_id,class,hash,size,ndim,...,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,is_registered,Modality,ROINames,BodyPartExamined
0,10,midRT,10_midRT,HNTSMRG24_train/10/midRT/10_midRT_T2.nii.gz,scan,10_midRT_10_midRT,MedImage,e8e14d254c581cd759fef50b0d54b126c24da149,"(512, 512, 76)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,0,MR,,OROPHARYNX
1,10,midRT,10_midRT,HNTSMRG24_train/10/midRT/10_midRT_mask.nii.gz,mask,10_midRT_10_midRT,Mask,2bdaf929ea9b48c200053952ea4a0f1875146d7e,"(512, 512, 76)",3,...,1.185530,14.438213,2619.610821,"(17.639697402822986, 33.93230573452438, 40.227...",2.0,HNTSMRG24_train/10/midRT/10_midRT_T2.nii.gz,0,SEG,"GTVp,GTVn",OROPHARYNX
2,10,preRT,10_preRT,HNTSMRG24_train/10/preRT/10_preRT_T2.nii.gz,scan,10_preRT_10_preRT,MedImage,83d8db42388c09b9b41c3f980ca87290a2668778,"(512, 512, 75)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,0,MR,,OROPHARYNX
3,10,preRT,10_preRT,HNTSMRG24_train/10/preRT/10_preRT_mask.nii.gz,mask,10_preRT_10_preRT,Mask,65566f1960f7b4bf71264b3442412063e151f6e8,"(512, 512, 75)",3,...,1.676271,14.746881,2732.815072,"(15.27453786193793, 31.654766112668213, 53.061...",2.0,HNTSMRG24_train/10/preRT/10_preRT_T2.nii.gz,0,SEG,"GTVp,GTVn",OROPHARYNX
4,101,midRT,101_midRT,HNTSMRG24_train/101/midRT/101_midRT_T2.nii.gz,scan,101_midRT_101_midRT,MedImage,afcd8a4d1bdc913d3c22fe232c8935762a639b77,"(512, 512, 73)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,0,MR,,OROPHARYNX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,95,midRT,95_preRT,HNTSMRG24_train/95/midRT/95_preRT_mask_registe...,mask,95_midRT_95_preRT_registered,Mask,ad44969cff59873b27926bed9c4b114d5b89c538,"(768, 768, 116)",3,...,1.772212,17.839432,3999.188850,"(24.385387992901403, 32.41856660529757, 57.452...",3.0,HNTSMRG24_train/95/midRT/95_preRT_T2_registere...,1,SEG,"GTVp,GTVn",OROPHARYNX
896,96,midRT,96_preRT,HNTSMRG24_train/96/midRT/96_preRT_T2_registere...,scan,96_midRT_96_preRT_registered,MedImage,4edcae7ef36c696019cddeec0e24bd6e507a8781,"(512, 512, 75)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,1,MR,,OROPHARYNX
897,96,midRT,96_preRT,HNTSMRG24_train/96/midRT/96_preRT_mask_registe...,mask,96_midRT_96_preRT_registered,Mask,3c4c62c1cd41c16956fd2af3aeed79e41aae0e01,"(512, 512, 75)",3,...,3.710939,8.743070,960.589361,"(7.704017074688655, 13.6754109042931, 50.74861...",2.0,HNTSMRG24_train/96/midRT/96_preRT_T2_registere...,1,SEG,"GTVp,GTVn",OROPHARYNX
898,99,midRT,99_preRT,HNTSMRG24_train/99/midRT/99_preRT_T2_registere...,scan,99_midRT_99_preRT_registered,MedImage,fc7cba8c4b17b047b6d86c5d11b3913f0d119f89,"(512, 512, 81)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,1,MR,,OROPHARYNX


In [39]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet

index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df["SampleID"] = index_df["SampleID"].astype(str)
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)

,patient_number,scan_time,name,filepath,file_type,SampleID,class,hash,size,ndim,...,mask.elongation,mask.equivalent_spherical_radius,mask.equivalent_spherical_perimeter,mask.equivalent_ellipsoid_diameters,mask.volume_count,reference_scan,is_registered,Modality,ROINames,BodyPartExamined
0,10,midRT,10_midRT,HNTSMRG24_train/10/midRT/10_midRT_T2.nii.gz,scan,10_midRT_10_midRT,MedImage,e8e14d254c581cd759fef50b0d54b126c24da149,"(512, 512, 76)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,0,MR,,OROPHARYNX
1,10,midRT,10_midRT,HNTSMRG24_train/10/midRT/10_midRT_mask.nii.gz,mask,10_midRT_10_midRT,Mask,2bdaf929ea9b48c200053952ea4a0f1875146d7e,"(512, 512, 76)",3,...,1.185530,14.438213,2619.610821,"(17.639697402822986, 33.93230573452438, 40.227...",2.0,HNTSMRG24_train/10/midRT/10_midRT_T2.nii.gz,0,SEG,"GTVp,GTVn",OROPHARYNX
2,10,preRT,10_preRT,HNTSMRG24_train/10/preRT/10_preRT_T2.nii.gz,scan,10_preRT_10_preRT,MedImage,83d8db42388c09b9b41c3f980ca87290a2668778,"(512, 512, 75)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,0,MR,,OROPHARYNX
3,10,preRT,10_preRT,HNTSMRG24_train/10/preRT/10_preRT_mask.nii.gz,mask,10_preRT_10_preRT,Mask,65566f1960f7b4bf71264b3442412063e151f6e8,"(512, 512, 75)",3,...,1.676271,14.746881,2732.815072,"(15.27453786193793, 31.654766112668213, 53.061...",2.0,HNTSMRG24_train/10/preRT/10_preRT_T2.nii.gz,0,SEG,"GTVp,GTVn",OROPHARYNX
4,101,midRT,101_midRT,HNTSMRG24_train/101/midRT/101_midRT_T2.nii.gz,scan,101_midRT_101_midRT,MedImage,afcd8a4d1bdc913d3c22fe232c8935762a639b77,"(512, 512, 73)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,0,MR,,OROPHARYNX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,95,midRT,95_preRT,HNTSMRG24_train/95/midRT/95_preRT_mask_registe...,mask,95_midRT_95_preRT_registered,Mask,ad44969cff59873b27926bed9c4b114d5b89c538,"(768, 768, 116)",3,...,1.772212,17.839432,3999.188850,"(24.385387992901403, 32.41856660529757, 57.452...",3.0,HNTSMRG24_train/95/midRT/95_preRT_T2_registere...,1,SEG,"GTVp,GTVn",OROPHARYNX
896,96,midRT,96_preRT,HNTSMRG24_train/96/midRT/96_preRT_T2_registere...,scan,96_midRT_96_preRT_registered,MedImage,4edcae7ef36c696019cddeec0e24bd6e507a8781,"(512, 512, 75)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,1,MR,,OROPHARYNX
897,96,midRT,96_preRT,HNTSMRG24_train/96/midRT/96_preRT_mask_registe...,mask,96_midRT_96_preRT_registered,Mask,3c4c62c1cd41c16956fd2af3aeed79e41aae0e01,"(512, 512, 75)",3,...,3.710939,8.743070,960.589361,"(7.704017074688655, 13.6754109042931, 50.74861...",2.0,HNTSMRG24_train/96/midRT/96_preRT_T2_registere...,1,SEG,"GTVp,GTVn",OROPHARYNX
898,99,midRT,99_preRT,HNTSMRG24_train/99/midRT/99_preRT_T2_registere...,scan,99_midRT_99_preRT_registered,MedImage,fc7cba8c4b17b047b6d86c5d11b3913f0d119f89,"(512, 512, 81)",3,...,NaN,NaN,NaN,NaN,NaN,NaN,1,MR,,OROPHARYNX
